In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Notebook 2 — Training & Evaluation\n", "TranAD model on NASA SMAP/MSL dataset"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import subprocess\n",
    "import sys\n",
    "import os\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import torch\n",
    "\n",
    "print(f'PyTorch version : {torch.__version__}')\n",
    "print(f'CUDA available  : {torch.cuda.is_available()}')\n",
    "if torch.cuda.is_available():\n",
    "    print(f'GPU             : {torch.cuda.get_device_name(0)}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run TranAD training on SMAP\n",
    "os.chdir('../Backend')\n",
    "result = subprocess.run(\n",
    "    [sys.executable, 'main.py', '--model', 'TranAD', '--dataset', 'SMAP', '--retrain'],\n",
    "    capture_output=True, text=True\n",
    ")\n",
    "print(result.stdout)\n",
    "if result.returncode != 0:\n",
    "    print('STDERR:', result.stderr)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Parse training output for loss values\n",
    "lines = result.stdout.split('\\n')\n",
    "losses = []\n",
    "for line in lines:\n",
    "    if 'L1 =' in line:\n",
    "        try:\n",
    "            val = float(line.split('L1 =')[-1].strip())\n",
    "            losses.append(val)\n",
    "        except:\n",
    "            pass\n",
    "\n",
    "if losses:\n",
    "    plt.figure(figsize=(10, 4))\n",
    "    plt.plot(losses, color='steelblue', linewidth=1.5)\n",
    "    plt.title('TranAD Training Loss — SMAP')\n",
    "    plt.xlabel('Epoch')\n",
    "    plt.ylabel('L1 Loss')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print('No loss values found in output — check training logs.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Parse final metrics from training output\n",
    "import json\n",
    "import re\n",
    "\n",
    "metrics = {}\n",
    "for line in lines:\n",
    "    if 'f1' in line.lower() and '{' in line:\n",
    "        try:\n",
    "            # extract dict-like structure\n",
    "            match = re.search(r'\\{.*\\}', line)\n",
    "            if match:\n",
    "                raw = match.group().replace(\"'\", '\"')\n",
    "                metrics = json.loads(raw)\n",
    "        except:\n",
    "            pass\n",
    "\n",
    "if metrics:\n",
    "    print('=== Model Evaluation Metrics ===')\n",
    "    for k, v in metrics.items():\n",
    "        print(f'  {k:20s}: {v}')\n",
    "else:\n",
    "    print('Metrics not parsed — check raw output above.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize anomaly scores on test set channel T-1\n",
    "TEST_DIR = './data/test'\n",
    "channel_file = os.path.join(TEST_DIR, 'T-1.npy')\n",
    "\n",
    "if os.path.exists(channel_file):\n",
    "    test_data = np.load(channel_file)\n",
    "    print(f'Test data shape: {test_data.shape}')\n",
    "\n",
    "    plt.figure(figsize=(14, 4))\n",
    "    plt.plot(test_data[:, 0], color='steelblue', linewidth=0.7, label='Telemetry')\n",
    "    plt.title('T-1 Test Channel — Raw Telemetry Values')\n",
    "    plt.xlabel('Timestep')\n",
    "    plt.ylabel('Value')\n",
    "    plt.legend()\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print('Test data not found — run download_data.sh first.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Precision / Recall / F1 bar chart\n",
    "if metrics:\n",
    "    keys = ['precision', 'recall', 'f1']\n",
    "    vals = [metrics.get(k, 0) for k in keys]\n",
    "\n",
    "    plt.figure(figsize=(6, 4))\n",
    "    bars = plt.bar(keys, vals, color=['#378ADD', '#1D9E75', '#EF9F27'], width=0.4)\n",
    "    for bar, val in zip(bars, vals):\n",
    "        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,\n",
    "                 f'{val:.3f}', ha='center', va='bottom', fontsize=11)\n",
    "    plt.ylim(0, 1.1)\n",
    "    plt.title('TranAD — Precision / Recall / F1 on SMAP')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print('Run training cell first to get metrics.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Labeled anomaly windows overlay\n",
    "labels = pd.read_csv('./data/labeled_anomalies.csv')\n",
    "t1_labels = labels[labels['chan_id'] == 'T-1']\n",
    "print(t1_labels[['chan_id', 'anomaly_sequences', 'spacecraft']])"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}